# 5.1 Теоретические и выборочные характеристики — практика

Конспект: [`README.md`](README.md). Здесь каждый термин и каждую формулу
отрабатываем руками на одних и тех же данных.

**Легенда**

| Значок | Что делать |
|---|---|
| 🧩 **Задание** | Напиши код в ячейке ниже, на месте `...` |
| ✅ **Проверка** | Запусти и ничего не меняй. Упала — ищи ошибку в своём решении |
| ❓ **Объясни** | Ответь текстом в ячейке ✍️. Это обязательная часть, а не украшение |
| 🚀 **Расширение** | По желанию: копнуть глубже |

**Правило игры.** `pop` — генеральная совокупность: все заказы маркетплейса за
месяц. В реальной работе её у тебя **нет**, есть только выборка. Здесь мы
используем `pop` ровно для двух вещей: **брать из неё выборки** и **сверять
оценки с истиной**. Всё остальное считаем только по выборкам.

Решай сверху вниз: задания опираются на предыдущие.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, integrate

from ds_course.marketplace import CATEGORIES, make_population, true_check_mean

pop = make_population()  # генеральная совокупность: 1 000 000 заказов
rng = np.random.default_rng(2026)
pop.head()

Поля: `category` — категория, `check` — сумма заказа (₽), `device` — `app`/`web`,
`region`, `delivery_days` — дней до доставки, `rating` — оценка 1–5,
`returned` — был ли возврат. Как именно сгенерированы данные («законы природы»
нашего мира), описано в `src/ds_course/marketplace.py`. Загляни туда, но
оценивать всё равно будем по выборкам.

## Часть 1. Задача оценивания

**Параметр** $\theta$ — средний чек по всем заказам месяца. Мы его «подсмотрим»
по всей совокупности, только чтобы было с чем сверяться.

In [ ]:
theta = pop["check"].mean()  # параметр. В жизни неизвестен!
theta

### 🧩 Задание 1.1. Случайная выборка

Напиши `take_sample`: **простую случайную выборку** `n` заказов **без
возвращения**. Вся случайность должна идти из переданного `rng`, тогда выборку
можно воспроизвести.

<details><summary>Подсказка</summary>

`rng.choice(len(pop), size=n, replace=False)` даёт номера строк, а `pop.iloc[...]`
выбирает строки по этим номерам.
</details>

In [ ]:
# 🧩 Задание 1.1
def take_sample(pop: pd.DataFrame, n: int, rng: np.random.Generator) -> pd.DataFrame:
    """Простая случайная выборка n заказов без возвращения."""
    ...  # ваш код

In [ ]:
# ✅ Проверка 1.1
s1 = take_sample(pop, 500, np.random.default_rng(1))
s2 = take_sample(pop, 500, np.random.default_rng(2))
s1_again = take_sample(pop, 500, np.random.default_rng(1))
assert isinstance(s1, pd.DataFrame) and len(s1) == 500, "нужен DataFrame из n строк"
assert s1.index.is_unique, "без возвращения: один заказ не может попасть дважды"
small = take_sample(pop.head(600), 600, np.random.default_rng(3))
assert small.index.is_unique and len(small) == 600, "без возвращения: 600 из 600 — это все заказы ровно по разу"
assert s1.equals(s1_again), "тот же rng-seed → та же выборка (воспроизводимость)"
assert not s1.index.equals(s2.index), "другой seed → другая выборка"
assert s1["order_id"].max() > 100_000, "выборка из всей совокупности, а не из её начала"
print("✅ 1.1 — верно")

### 🧩 Задание 1.2. Оценка — случайная величина

Возьми **5 разных** выборок по 500 заказов. По каждой посчитай оценку
среднего чека $\hat\theta = \overline X$. Сложи пять оценок в список `estimates_5`.

In [ ]:
# 🧩 Задание 1.2
estimates_5 = ...  # ваш код: список из 5 чисел
estimates_5

In [ ]:
# ✅ Проверка 1.2
assert len(estimates_5) == 5
assert len(set(estimates_5)) == 5, "пять РАЗНЫХ выборок дают пять разных оценок"
assert all(0.7 * theta < e < 1.3 * theta for e in estimates_5)
print("θ =", round(theta, 1), "| оценки:", np.round(estimates_5, 1))
print("✅ 1.2 — верно")

❓ **Объясни.** Что здесь случайно, а что нет: $\theta$, наблюдения $X_i$,
оценка $\hat\theta$? Почему оценки разные, а $\theta$ одно? Назови «стол,
линейку и замеры» из урока в терминах этого примера.

✍️ **Твой ответ:**

…

## Часть 2. Свойства оценок

Свойства оценки — это свойства её **распределения**: как она ведёт себя
на всех возможных выборках сразу. Увидеть это распределение можно методом
**Монте-Карло**: берём много выборок и смотрим на облако оценок.

Чтобы было быстро, сделаем это векторно. Матрица индексов размера `(n_sim, n)`:
каждая строка — отдельная выборка. Индексы берём **с возвращением**
(`rng.integers`): при $n \ll N$ разница с выборкой без возвращения ничтожна,
зато это на порядки быстрее.

### 🧩 Задание 2.1. Машина Монте-Карло

`estimator` получает матрицу `(n_sim, n)` и возвращает вектор из `n_sim` оценок,
например `lambda s: s.mean(axis=1)`.

In [ ]:
# 🧩 Задание 2.1
def simulate(values: np.ndarray, estimator, n: int, n_sim: int, rng: np.random.Generator) -> np.ndarray:
    """n_sim независимых выборок размера n из values → вектор из n_sim оценок."""
    ...  # ваш код

In [ ]:
# ✅ Проверка 2.1
check_values = pop["check"].to_numpy()
est = simulate(check_values, lambda s: s.mean(axis=1), n=500, n_sim=2000, rng=np.random.default_rng(0))
assert est.shape == (2000,)
assert est.std() > 0, "оценки по разным выборкам должны различаться"
assert abs(est.mean() - theta) < 0.02 * theta, "среднее оценок должно быть близко к θ"
# забегая вперёд (урок 3): разброс среднего ≈ σ/√n — если строки независимы
assert 0.85 < est.std() / (check_values.std() / np.sqrt(500)) < 1.15, "строки должны быть независимыми выборками"
print("✅ 2.1 — верно")

In [ ]:
plt.hist(est, bins=50, density=True, alpha=0.7)
plt.axvline(theta, color="k", ls="--", label="θ (параметр)")
plt.title("Распределение оценки X̄ по 2000 выборкам, n=500")
plt.legend();

### 🧩 Задание 2.2. Несмещённость: почему делят на $n-1$

Возьмём срок доставки: у него конечный разброс без длинного хвоста, и шум
Монте-Карло не мешает. Маленькое $n = 5$ сделает смещение заметным.

Посчитай среднее по 100 000 выборок для двух оценок дисперсии:
- `mean_S2` — для $S^2$ (делитель $n$);
- `mean_s2` — для $s^2$ (делитель $n-1$).

Сравни с теорией: $M[S^2] = \frac{n-1}{n}\sigma^2$.

In [ ]:
days = pop["delivery_days"].to_numpy()
sigma2 = days.var()  # теоретическая дисперсия (по всей совокупности)
sigma2

In [ ]:
# 🧩 Задание 2.2
mean_S2 = ...  # ваш код
mean_s2 = ...  # ваш код
mean_S2 / sigma2, mean_s2 / sigma2

In [ ]:
# ✅ Проверка 2.2
assert 0.78 < mean_S2 / sigma2 < 0.82, "M[S²] = (n−1)/n · σ² = 0.8σ² при n=5"
assert 0.98 < mean_s2 / sigma2 < 1.02, "s² несмещённая: M[s²] = σ²"
print("✅ 2.2 — верно")

❓ **Объясни.** Почему $S^2$ систематически занижает дисперсию? Подсказка: от
чего мы считаем отклонения — от $\mu$ или от $\overline X$? И что ближе к точкам
выборки?

✍️ **Твой ответ:**

…

🚀 **Расширение.** Докажи на бумаге $M[S^2] = \frac{n-1}{n}\sigma^2$. Ход:
$\sum(X_i-\overline X)^2 = \sum(X_i-\mu)^2 - n(\overline X-\mu)^2$, а
$D[\overline X] = \sigma^2/n$. Затем проверь численно, как отношение
`mean_S2 / sigma2` зависит от $n$ при $n = 2, 3, 5, 10, 50$.

### 🧩 Задание 2.3. Состоятельность и «глупая» оценка

Состоятельность: $P(|\hat\theta_n - \theta| > \varepsilon) \to 0$ при $n\to\infty$.
Проверим это напрямую. Возьмём $\varepsilon$ = 5% от $\theta$ и для каждого $n$
оценим вероятность промаха как долю симуляций, где промах больше $\varepsilon$.

Три оценки параметра $\theta$ (среднего чека):
1. выборочное среднее;
2. «глупая» оценка: всегда 1500 ₽;
3. **выборочная медиана**.

In [ ]:
ns = [10, 30, 100, 300, 1000, 3000]
eps = 0.05 * theta
check_values = pop["check"].to_numpy()

est_mean = lambda s: s.mean(axis=1)
est_dumb = lambda s: np.full(len(s), 1500.0)
est_median = lambda s: np.median(s, axis=1)

In [ ]:
# 🧩 Задание 2.3
def prob_miss(estimator, n: int, n_sim: int = 2000) -> float:
    """Оценка P(|θ̂_n − θ| > eps) методом Монте-Карло."""
    ...  # ваш код

p_mean = [prob_miss(est_mean, n) for n in ns]
p_dumb = [prob_miss(est_dumb, n) for n in ns]
p_median = [prob_miss(est_median, n) for n in ns]

In [ ]:
# ✅ Проверка 2.3
assert len(p_mean) == len(ns)
assert p_mean[0] > 0.6 and p_mean[-1] < 0.05, "среднее: при малом n промахи часты, при большом — редки"
assert all(b <= a + 0.03 for a, b in zip(p_mean, p_mean[1:])), "вероятность промаха должна падать с ростом n"
assert all(p == 1.0 for p in p_dumb), "глупая оценка промахивается всегда — данные ей не помогают"
assert p_median[-1] > 0.95, "медиана сходится, но НЕ к среднему чеку"
print("✅ 2.3 — верно")

In [ ]:
plt.semilogx(ns, p_mean, "o-", label="выборочное среднее")
plt.semilogx(ns, p_dumb, "s--", label="«всегда 1500»")
plt.semilogx(ns, p_median, "^-", label="выборочная медиана")
plt.xlabel("n"); plt.ylabel("P(|θ̂ − θ| > ε)"); plt.legend()
plt.title("Состоятельность как оценка СРЕДНЕГО чека");

❓ **Объясни.** Медиана — хорошая, состоятельная оценка. Почему здесь она
промахивается всё чаще с ростом $n$? К какому числу она сходится? Сверь с
`pop["check"].median()`. Сформулируй правило: «состоятельна **для чего**».

✍️ **Твой ответ:**

…

### 🧩 Задание 2.4. Эффективность и разложение MSE

Нужен параметр, для которого среднее и медиана **обе** несмещённые и
состоятельные. Возьмём $\ln$ чека в категории «Одежда». По построению данных он
распределён нормально, то есть симметрично, и центр у него один:
$\theta_{\ln}$.

1. Напиши `bias`, `variance` (дисперсия оценки, делитель — число симуляций) и
   `mse`. **`mse` считай напрямую по определению** $M[(\hat\theta-\theta)^2]$,
   а не через первые две функции. Иначе нечего будет проверять.
2. Получи облака оценок `est_mean_log` и `est_median_log`: `n=50`,
   `n_sim=20_000`.

In [ ]:
log_cloth = np.log(pop.loc[pop["category"] == "Одежда", "check"].to_numpy())
theta_log = log_cloth.mean()
theta_log

In [ ]:
# 🧩 Задание 2.4
def bias(estimates, theta): ...
def variance(estimates): ...
def mse(estimates, theta): ...

est_mean_log = ...    # ваш код
est_median_log = ...  # ваш код

In [ ]:
# ✅ Проверка 2.4
for e in (est_mean_log, est_median_log, np.full(10, 7.0)):
    assert np.isclose(mse(e, theta_log), variance(e) + bias(e, theta_log) ** 2, rtol=1e-9, atol=0), \
        "MSE = D + bias² должно выполняться точно (подсказка: делитель в variance)"
assert abs(bias(est_mean_log, theta_log)) < 0.01 and abs(bias(est_median_log, theta_log)) < 0.01, \
    "для симметричного распределения обе оценки несмещённые"
ratio = mse(est_median_log, theta_log) / mse(est_mean_log, theta_log)
assert 1.35 < ratio < 1.75, f"MSE медианы / MSE среднего = {ratio:.2f}, ожидаем ≈ π/2"
print(f"MSE(медиана) / MSE(среднее) = {ratio:.3f}  (π/2 = {np.pi / 2:.3f})")
print("✅ 2.4 — верно")

❓ **Объясни.** Отношение $\approx \pi/2 \approx 1{,}57$. Что это значит на
практике? Подсказка: дисперсия оценки $\propto 1/n$. Сколько нужно наблюдений
медиане, чтобы сравняться по точности со средним на 1000 наблюдений? И почему
в Части 2.3 медиана всё равно была плохой оценкой?

✍️ **Твой ответ:**

…

🚀 **Расширение: смещённая оценка, которая лучше.** Рассмотри $c\cdot\overline X$
для $\theta$ = среднего чека при $n=10$. При каком $c$ MSE минимальна?
Построй MSE($c$) для $c \in [0.7, 1.1]$. Объясни, почему оптимум $c<1$,
то есть смещённая оценка точнее несмещённой. Это и есть компромисс bias–variance,
он встретится в ML.

## Часть 3. Репрезентативность: объём не лечит перекос

Продакт говорит: «Зачем случайная выборка? У нас есть данные по 10 000
заказов из приложения, это же много».

In [ ]:
# 🧩 Задание 3.1
n = 10_000
se = pop["check"].std() / np.sqrt(n)  # типичная ошибка СЛУЧАЙНОЙ выборки такого размера

mean_app = ...     # ваш код: средний чек по случайной выборке n заказов ТОЛЬКО из приложения
mean_random = ...  # ваш код: средний чек по случайной выборке n из ВСЕЙ совокупности
(mean_app - theta) / se, (mean_random - theta) / se

In [ ]:
# ✅ Проверка 3.1
assert not np.isclose(mean_random, theta, rtol=0, atol=1e-9), "mean_random посчитан по всей совокупности, а нужна выборка"
assert not np.isclose(mean_app, pop.loc[pop["device"] == "app", "check"].mean(), rtol=0, atol=1e-9), \
    "mean_app посчитан по всем заказам из приложения, а нужна выборка из них"
assert abs(mean_random - theta) < 4 * se, "случайная выборка: промах в пределах нескольких se"
assert (mean_app - theta) / se > 5, "выборка из приложения должна систематически завышать средний чек"
print("✅ 3.1 — верно")

❓ **Объясни.**
1. Почему промах выборки из приложения во много раз больше `se`? Станет ли
   лучше при $n$ = 100 000?
2. **Почему** пользователи приложения тратят больше? Загляни в
   `pd.crosstab(pop["category"], pop["device"], normalize="columns")`.
3. Какое из двух условий репрезентативности здесь нарушено?

✍️ **Твой ответ:**

…

🚀 **Расширение.** Объясни смещение через состав выборки:
$\overline X_{app} = \sum_k w^{app}_k \cdot \overline X_k$, где $w_k$ — доли
категорий. Перевзвесь выборку из приложения долями категорий во всей
совокупности (post-stratification). Насколько уменьшилось смещение и почему
оно не ушло совсем?

## Часть 4. Теоретические ↔ выборочные характеристики

### 🧩 Задание 4.1. Формулы руками

Можно использовать только `sum`, `len`, арифметику и `np.sqrt`. Никаких
`np.mean` и `np.var`: смысл в том, чтобы формула прошла через пальцы.

In [ ]:
# 🧩 Задание 4.1
def my_mean(x): ...
def my_var(x, ddof=0): ...   # делитель n − ddof
def my_std(x, ddof=0): ...
def my_proportion(mask): ... # частота события: m / n, mask — массив True/False

In [ ]:
# ✅ Проверка 4.1
_r = np.random.default_rng(11)
for _ in range(20):
    x = _r.normal(_r.uniform(-5, 5), _r.uniform(0.1, 10), size=_r.integers(2, 40))
    assert np.isclose(my_mean(x), np.mean(x))
    for d in (0, 1):
        assert np.isclose(my_var(x, ddof=d), np.var(x, ddof=d)), f"my_var при ddof={d}"
        assert np.isclose(my_std(x, ddof=d), np.std(x, ddof=d)), f"my_std при ddof={d}"
    mask = x > 0
    assert np.isclose(my_proportion(mask), mask.mean())
print("✅ 4.1 — верно")

### 🧩 Задание 4.2. Ловушка `ddof`

Одна и та же выборка из 20 чеков, два «одинаковых» вызова. **Не вычисляя
дисперсий**, предскажи формулой через $n$, во сколько раз `pd_var` больше `np_var`.

In [ ]:
# 🧩 Задание 4.2
x20 = take_sample(pop, 20, rng)["check"].to_numpy()
np_var = np.var(x20)
pd_var = pd.Series(x20).var()

predicted_ratio = ...  # ваш код: формула через n = len(x20)
pd_var / np_var, predicted_ratio

In [ ]:
# ✅ Проверка 4.2
assert np.isclose(pd_var / np_var, predicted_ratio), "у какой библиотеки какой ddof по умолчанию?"
print("✅ 4.2 — верно")

### 🧩 Задание 4.3. Три уровня «истины»

У среднего чека в «Одежде» три разных «истинных» значения, и полезно их не путать:

1. **Параметр модели (распределения).** Данные сгенерированы из логнормального
   закона с известными $\mu, \sigma$. Отсюда $E[X] = e^{\mu + \sigma^2/2}$
   и это же число как интеграл $\int_0^\infty x f(x)\,dx$.
2. **Параметр конечной совокупности** — среднее по всем ~350 000 заказам
   «Одежды». Именно это «средняя зарплата по Самаре» из урока.
3. **Оценка по выборке** — среднее 500 случайных заказов.

<details><summary>Подсказка про scipy</summary>

У `stats.lognorm` неочевидная параметризация: `s=σ`, `scale=exp(μ)`.
Плотность: `stats.lognorm(s=..., scale=...).pdf(x)`. Интеграл:
`integrate.quad(lambda x: x * f(x), 0, np.inf)[0]`.
</details>

In [ ]:
c = CATEGORIES["Одежда"]
cloth = pop[pop["category"] == "Одежда"]
c

In [ ]:
# 🧩 Задание 4.3
mu_analytic = ...    # ваш код: формула exp(mu + sigma²/2)
mu_integral = ...    # ваш код: ∫ x f(x) dx численно
mu_population = ...  # ваш код: среднее по всей «Одежде»
mu_sample = ...      # ваш код: среднее по случайной выборке 500 заказов «Одежды»
mu_analytic, mu_integral, mu_population, mu_sample

In [ ]:
# ✅ Проверка 4.3
assert np.isclose(mu_analytic, true_check_mean("Одежда"))
assert np.isclose(mu_integral, mu_analytic, rtol=1e-6), "интеграл должен совпасть с формулой (проверь параметризацию lognorm)"
assert abs(mu_population / mu_analytic - 1) < 0.01, "совокупность большая — её среднее близко к параметру модели"
assert not np.isclose(mu_sample, mu_population, rtol=0, atol=1e-9), "mu_sample — по выборке, а не по всей «Одежде»"
assert abs(mu_sample / mu_analytic - 1) < 0.15
print("✅ 4.3 — верно")

❓ **Объясни.** Какой из трёх уровней доступен аналитику в реальной жизни?
Почему 1-й и 2-й уровни почти совпадают, а 3-й заметно отличается?

✍️ **Твой ответ:**

…

### 🧩 Задание 4.4. Дискретная величина и доля

Для дискретной случайной величины интеграл превращается в сумму:
$E[X] = \sum_k x_k p_k$. Доля события $A$ оценивается частотой:
$\hat p = m_A / n$.

In [ ]:
# 🧩 Задание 4.4
p_k = ...       # ваш код: Series, индекс — оценка 1..5, значение — её вероятность в совокупности
E_rating = ...  # ваш код: Σ k · p_k
p_return = ...  # ваш код: теоретическая доля возвратов (по совокупности)
p_hat = ...     # ваш код: частота возвратов m/n в случайной выборке n = 1000
p_k, E_rating, p_return, p_hat

In [ ]:
# ✅ Проверка 4.4
assert list(p_k.index) == [1, 2, 3, 4, 5], "индекс — оценки 1..5 по возрастанию"
assert np.isclose(p_k.sum(), 1), "вероятности в сумме дают 1"
assert np.isclose(E_rating, pop["rating"].mean()), "Σ k·p_k по совокупности = среднее по совокупности"
assert np.isclose(p_return, pop["returned"].mean())
assert not np.isclose(p_hat, p_return, rtol=0, atol=1e-12), "p_hat — частота в выборке, а не доля в совокупности"
assert abs(p_hat - p_return) < 0.05
print("✅ 4.4 — верно")

## Часть 5. Вариационный ряд, ECDF, квантили

### 🧩 Задание 5.1. Выборочная функция распределения

$\hat F_n(x) = \dfrac{\#\{i: X_i \le x\}}{n}$. Аргумент `x` может быть числом
или массивом. Сделай векторно.

<details><summary>Подсказка</summary>

Отсортируй выборку (это вариационный ряд) и используй `np.searchsorted`.
Внимательно с параметром `side`: неравенство **нестрогое**.
</details>

In [ ]:
# 🧩 Задание 5.1
def ecdf(sample, x):
    """F̂_n(x) — доля элементов выборки, не превосходящих x."""
    ...  # ваш код

In [ ]:
# ✅ Проверка 5.1
d50 = take_sample(pop, 50, np.random.default_rng(5))["delivery_days"].to_numpy()  # много повторов
pts = np.concatenate([np.arange(-1, 15, 0.5), d50])
assert np.allclose(ecdf(d50, pts), stats.ecdf(d50).cdf.evaluate(pts)), "в точках наблюдений тоже должно совпадать"
assert ecdf(d50, d50.min() - 1) == 0 and ecdf(d50, d50.max()) == 1
v, cnt = np.unique(d50, return_counts=True)
jumps = ecdf(d50, v) - ecdf(d50, v - 1e-9)
assert np.allclose(jumps, cnt / 50), "высота ступеньки = (число повторов) / n"
print("✅ 5.1 — верно")

In [ ]:
F_pop = lambda x: ecdf(check_values, x)  # «теоретическая» F — по всей совокупности
grid = np.linspace(0, 8000, 1000)
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, n_ in zip(axes, [20, 200, 2000]):
    s = take_sample(pop, n_, rng)["check"].to_numpy()
    ax.step(grid, ecdf(s, grid), where="post", label=f"F̂_n, n={n_}")
    ax.plot(grid, F_pop(grid), "k--", lw=1, label="F (совокупность)")
    ax.set_title(f"n = {n_}"); ax.legend()

### 🧩 Задание 5.2. Насколько ECDF далека от истины

Расстояние Колмогорова: $D_n = \sup_x |\hat F_n(x) - F(x)|$.
Теорема Гливенко–Кантелли: $D_n \to 0$.

<details><summary>Подсказка: где искать sup</summary>

Между соседними точками выборки $\hat F_n$ постоянна, а $F$ растёт. Поэтому
максимум отклонения достигается в точках выборки: **справа** от скачка
(значение $\hat F_n(X_{(k)})$) или **слева** (предел $\hat F_n(X_{(k)}-0)$).
Значения $F$ справа и слева от точки тоже надо брать свои:
`side="right"` и `side="left"` в `searchsorted`.
</details>

In [ ]:
# 🧩 Задание 5.2
def sup_distance(sample, population) -> float:
    """D_n = max_x |F̂_n(x) − F(x)|, где F — функция распределения совокупности."""
    ...  # ваш код

D = {n_: sup_distance(take_sample(pop, n_, rng)["check"].to_numpy(), check_values) for n_ in [20, 200, 2000]}
D

In [ ]:
# ✅ Проверка 5.2
_r = np.random.default_rng(3)
for n_ in (5, 20, 200):
    s = _r.choice(check_values, n_, replace=False)
    assert np.isclose(sup_distance(s, check_values), stats.ks_2samp(s, check_values).statistic, atol=1e-9), \
        f"n={n_}: не совпало с ks_2samp — учтены ли значения СЛЕВА от скачков?"
assert D[20] > D[2000] and D[2000] < 0.05
print("✅ 5.2 — верно")

🚀 **Расширение.** Неравенство Дворецкого–Кифера–Вольфовица:
$P(D_n > \varepsilon) \le 2e^{-2n\varepsilon^2}$. Проверь его Монте-Карло
при $n=200$, $\varepsilon = 0{,}05$. Затем обрати: какое $n$ гарантирует
$D_n \le 0{,}02$ с вероятностью 95%?

### 🧩 Задание 5.3. Квантиль по формуле курса

$\hat x_\alpha = X_{([n\alpha]+1)}$: в нумерации с единицы. В Python индексы
с нуля, учти это.

⚠️ Посчитай в отдельной ячейке `100 * 0.29` и `int(100 * 0.29)`. Твоя функция
обязана давать правильный ответ и в этом случае.

In [ ]:
# 🧩 Задание 5.3
def course_quantile(sample, alpha: float) -> float:
    """Выборочная квантиль уровня alpha по формуле X_([nα]+1)."""
    ...  # ваш код

In [ ]:
# ✅ Проверка 5.3
x100 = np.random.default_rng(8).normal(size=100)
xs = np.sort(x100)
assert course_quantile(x100, 0.9) == xs[90], "n=100, α=0.9 → X_(91) (в нумерации с единицы)"
assert course_quantile(x100, 0.5) == xs[50]
assert course_quantile(x100, 0.29) == xs[29], "100·0.29 — это 29, а не 28.999…"
assert course_quantile(x100, 0.57) == xs[57]
assert course_quantile(x100, 0.0) == xs[0]
assert course_quantile(np.arange(7), 0.5) == 3, "медиана из 7 элементов — 4-й по счёту"
for a in (0.123, 0.456, 0.789):  # nα нецелое
    assert course_quantile(x100, a) == np.quantile(x100, a, method="inverted_cdf")
print("✅ 5.3 — верно")

❓ **Объясни.** Сравни для `x100` и `α = 0.9` три ответа:
`course_quantile`, `np.quantile(..., method="inverted_cdf")` и `np.quantile(...)`
по умолчанию. Почему они разные? Сформулируй через $\hat F_n$ (для выборки
без повторов, как `x100`): формула курса —
наименьший $X_{(k)}$, у которого $\hat F_n > \alpha$, а `inverted_cdf` — у
которого $\hat F_n \ge \alpha$. Важно ли это различие при $n$ = 100 000?

✍️ **Твой ответ:**

…

## Часть 6. Оценка плотности

### 🧩 Задание 6.1. Гистограмма: площадь, а не высота

Высота столбика $h_j = \dfrac{n_j}{n\,\Delta_j}$. Интервалы **неравные**,
так и задумано. Счётчики $n_j$ можно взять из `np.histogram(x, bins=edges)[0]`,
а нормировку сделай сам.

In [ ]:
# 🧩 Задание 6.1
def density_histogram(x, edges):
    """Высоты столбиков h_j = n_j / (n Δ_j)."""
    ...  # ваш код

In [ ]:
# ✅ Проверка 6.1
x5k = take_sample(pop, 5000, np.random.default_rng(4))["check"].to_numpy()
edges = np.array([x5k.min(), 500, 1000, 1500, 2000, 3000, 5000, 10000, x5k.max()])
h = density_histogram(x5k, edges)
assert np.allclose(h, np.histogram(x5k, bins=edges, density=True)[0]), "высота должна учитывать ширину интервала Δ_j"
assert np.isclose(np.sum(h * np.diff(edges)), 1), "суммарная площадь столбиков = 1"
print("✅ 6.1 — верно")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharex=True)
w = np.diff(edges)
axes[0].bar(edges[:-1], np.histogram(x5k, bins=edges)[0] / len(x5k), width=w, align="edge", edgecolor="k")
axes[0].set_title("НЕВЕРНО: высота = n_j / n (широкие интервалы раздуты)")
axes[1].bar(edges[:-1], h, width=w, align="edge", edgecolor="k")
axes[1].set_title("ВЕРНО: высота = n_j / (n Δ_j), площадь = доля")
axes[1].set_xlim(0, 12000);

### 🧩 Задание 6.2. Ядра

Реализуй четыре ядра из урока **векторно** (`u` — массив). Требования к ядру:
$K(u) \ge 0$, $K(-u) = K(u)$, $\int K = 1$. Проверка будет требовать
именно их.

In [ ]:
# 🧩 Задание 6.2
def k_uniform(u): ...       # 1/2 при |u| ≤ 1
def k_triangular(u): ...    # 1 − |u| при |u| ≤ 1
def k_epanechnikov(u): ...  # 3/4 (1 − u²) при |u| ≤ 1
def k_gauss(u): ...         # стандартная нормальная плотность

KERNELS = {"uniform": k_uniform, "triangular": k_triangular, "epanechnikov": k_epanechnikov, "gauss": k_gauss}

In [ ]:
# ✅ Проверка 6.2
u = np.linspace(-4, 4, 801)
for name, K in KERNELS.items():
    vals = K(u)
    assert np.shape(vals) == u.shape, f"{name}: ядро должно работать с массивом"
    assert np.all(vals >= 0), f"{name}: K ≥ 0"
    assert np.allclose(vals, K(-u)), f"{name}: K чётное"
    area = integrate.quad(lambda t: float(K(np.array(t))), -10, 10, points=[-1, 0, 1], limit=200)[0]
    assert np.isclose(area, 1, atol=1e-6), f"{name}: площадь = {area:.4f}, а должна быть 1"
assert np.allclose(k_gauss(u), stats.norm.pdf(u))
print("✅ 6.2 — верно")

### 🧩 Задание 6.3. KDE по формуле

$\hat f_n(x) = \dfrac{1}{nh}\sum_{i=1}^n K\!\left(\dfrac{x - X_i}{h}\right)$.
Сделай векторно через broadcasting: матрица `(len(x), n)`.

In [ ]:
# 🧩 Задание 6.3
def kde(sample, x, h: float, kernel=k_gauss):
    """Ядерная оценка плотности в точках x."""
    ...  # ваш код

In [ ]:
# ✅ Проверка 6.3
s200 = np.log(take_sample(pop, 200, np.random.default_rng(6))["check"].to_numpy())
xg = np.linspace(s200.min() - 3, s200.max() + 3, 3001)
h_ = 0.2
# scipy: bw_method — это не h, а множитель: h = bw_method · s (ddof=1)
ref = stats.gaussian_kde(s200, bw_method=h_ / s200.std(ddof=1))(xg)
assert np.allclose(kde(s200, xg, h_), ref, rtol=1e-6), "гауссово KDE должно совпасть со scipy"
for name, K in KERNELS.items():
    area = np.trapezoid(kde(s200, xg, h_, K), xg)
    assert np.isclose(area, 1, atol=1e-3), f"{name}: площадь под KDE = {area:.3f}"
print("✅ 6.3 — верно")

### 🧩 Задание 6.4. Ширина полосы $h$ решает всё

Логарифм чека в «Одежде» распределён как $N(\mu, \sigma)$, поэтому **истинную
плотность мы знаем** и можем честно измерить ошибку KDE:
$\mathrm{ISE}(h) = \int (\hat f_h(x) - f(x))^2\,dx$. Посчитай ISE для каждого $h$
из сетки и найди лучшее.

In [ ]:
x_log = np.log(take_sample(cloth, 100, rng)["check"].to_numpy())
grid_log = np.linspace(c.mu - 5 * c.sigma, c.mu + 5 * c.sigma, 2001)
f_true = stats.norm(c.mu, c.sigma).pdf(grid_log)
hs = np.geomspace(0.005, 2, 60)
h_silverman = 1.06 * x_log.std(ddof=1) * len(x_log) ** (-1 / 5)  # правило Сильвермана

In [ ]:
# 🧩 Задание 6.4
ise = ...    # ваш код: массив ISE(h) для каждого h из hs (np.trapezoid по grid_log)
h_opt = ...  # ваш код: h с минимальной ISE
h_opt, h_silverman

In [ ]:
# ✅ Проверка 6.4
ise_ = np.asarray(ise)
assert ise_.shape == hs.shape
assert h_opt == hs[np.argmin(ise_)], "h_opt — это h, при котором ISE МИНИМАЛЬНА"
assert 1 / 3 < h_opt / h_silverman < 3, "оптимум должен быть одного порядка с правилом Сильвермана"
assert ise_[0] > 5 * ise_.min() and ise_[-1] > 2 * ise_.min(), "и слишком малое, и слишком большое h хуже оптимума"
print("✅ 6.4 — верно")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].loglog(hs, ise); axes[0].axvline(h_opt, color="k", ls="--", label="h_opt")
axes[0].axvline(h_silverman, color="g", ls=":", label="Сильверман"); axes[0].set_xlabel("h"); axes[0].set_ylabel("ISE"); axes[0].legend()
axes[1].plot(grid_log, f_true, color="gray", lw=3, label="истинная f")
for h_bw, col in [(0.03, "r"), (h_opt, "k"), (1.0, "g")]:
    axes[1].plot(grid_log, kde(x_log, grid_log, h_bw), col, label=f"h = {h_bw:.2f}")
axes[1].plot(x_log, np.zeros_like(x_log), "|k", ms=10); axes[1].legend();

❓ **Объясни.** Что видно при $h = 0{,}03$ и при $h = 1$? Как это связано с
«недосглаживанием» и «пересглаживанием» из урока? Почему в жизни ISE
посчитать нельзя и подбирают $h$ правилами вроде Сильвермана или кросс-валидацией?

✍️ **Твой ответ:**

…

🚀 **Расширение.** Повтори 6.4 с ядром Епанечникова и с прямоугольным.
Сильно ли меняется минимальная ISE по сравнению с гауссовым ядром? Это и есть
тезис урока: выбор ядра влияет слабее, чем выбор $h$. Ещё вариант: построй KDE
**самого чека** (не логарифма) и посмотри, что происходит у нуля и в хвосте.

## Итог

Ответь своими словами, без формул там, где можно:

1. Параметр, наблюдения, оценка: что из этого случайно?
2. Несмещённость, состоятельность, эффективность — по одной фразе и по примеру
   из этой тетрадки.
3. Почему `np.var` ≠ `pd.Series.var`, и когда это важно?
4. Когда большой объём выборки **не спасает**?
5. ECDF, гистограмма, KDE: что каждая из них оценивает, и в чём слабость каждой?

**Дальше.** $\overline X$ — случайная величина с разбросом $\approx \sigma/\sqrt n$
(это мы уже увидели в 2.1). Отсюда растут доверительные интервалы и расчёт
объёма выборки. А длинный правый хвост чеков разберём в уроке про выбросы.

✍️ **Твой ответ:**

…